## Prediction for Step 6 (from reading the scripts, before running)
- `astral` may not be installed (not in the documented dependencies).
- config.py paths are placeholders ("path/to/..."), so they must be overridden for the test.
- weather_match joins on "Datetime" but the weather table's column is "datetime": expect KeyError.
- If worked around: weather has only the 16:00 and 17:00 hours; the matcher rounds to the NEAREST hour, so 1,148 clips match 17:00 and 599 clips (17:30 onward) round to 18:00 and get blank weather.
- Step 6 ignores run_pipeline's folders and does not write outputs/weather.csv.
- Run as-is: weather_match() should raise KeyError: 'Datetime' (the weather table has 'datetime').
- After working around that: 1,148 clips match 17:00; 599 clips round to 18:00 and get blank weather; the 16:00 row is never used.

In [1]:
from nb_paths import PIPELINE_DIR, KWF_TEST_DIR
import sys, importlib.metadata as md
from pathlib import Path
import pandas as pd

HOME = Path.home()
RAW_DIR      = KWF_TEST_DIR / "raw_audio"
CLIPS_DIR    = KWF_TEST_DIR / "split_clips"
OUTPUTS_DIR  = KWF_TEST_DIR / "outputs"
sys.path.insert(0, str(PIPELINE_DIR))
sys.path.insert(0, str(PIPELINE_DIR / "scripts"))

try:
    print("astral", md.version("astral"))
except md.PackageNotFoundError:
    print("astral NOT INSTALLED")

import config
for name in ["AUDIO_MOTH_FOLDER", "WEATHER_DATA_PATH", "WEATHER_OUTPUT_CSV",
             "METADATA_OUTPUT_CSV", "LATITUDE", "LONGITUDE", "TIMEZONE"]:
    print(f"{name:22s} {getattr(config, name)}")

try:
    import weather_info, weather_matching
    print("\nImports OK")
except Exception as e:
    print("\nIMPORT PROBLEM:", type(e).__name__, e)

astral 3.2
AUDIO_MOTH_FOLDER      path/to/raw
WEATHER_DATA_PATH      path/to/weather_data.csv
WEATHER_OUTPUT_CSV     path/to/weather_matches.csv
METADATA_OUTPUT_CSV    path/to/metadata.csv
LATITUDE               8.6602
LONGITUDE              -83.6502
TIMEZONE               America/Costa_Rica

Imports OK


What this does: checks that astral is installed, prints the config.py values Step 6 depends on, and tries to import both weather scripts.

How to read the output: “NOT INSTALLED” for astral means the import will fail and the pipeline would crash on a fresh machine. The config lines show which values are still placeholders (path/to/...). “Imports OK” means the scripts load, and any error shows exactly what’s missing.

In [2]:
%pip install astral

Note: you may need to restart the kernel to use updated packages.


In [3]:
import importlib.metadata as md
print("astral", md.version("astral"))
try:
    import weather_info, weather_matching
    print("Imports OK")
except Exception as e:
    print("IMPORT PROBLEM:", type(e).__name__, e)

astral 3.2
Imports OK


In [4]:
import time

weather_hourly_csv = OUTPUTS_DIR / "nb06_weather_hourly.csv"

t0 = time.time()
weather_info.get_weather_info_all(
    audio_folder=str(RAW_DIR),
    output_csv=str(weather_hourly_csv),
)
print(f"Runtime: {time.time() - t0:.1f}s")

w = pd.read_csv(weather_hourly_csv)
print("\nShape:", w.shape)
print("Columns:", w.columns.tolist())
print(w[["Datetime", "Temperature", "precipitation", "Humidity", "weather_desc"]].to_string(index=False))

Processing: 20250319_163237.WAV
[OK] Combined weather data saved to ~/kwf_test/outputs/nb06_weather_hourly.csv (3 hours)
Runtime: 0.8s

Shape: (3, 11)
Columns: ['Datetime', 'Temperature', 'Windspeed', 'precipitation', 'Humidity', 'weathercode', 'weather_desc', 'timestamp_local', 'sunrise', 'sunset', 'time_of_day']
           Datetime  Temperature  precipitation  Humidity weather_desc
2025-03-19 16:00:00         27.8            0.0        76    Clear sky
2025-03-19 17:00:00         27.1            0.0        80    Clear sky
2025-03-19 18:00:00         26.1            0.0        86    Clear sky


What this does: calls get_weather_info_all with the raw recordings folder and an output file, the way run_pipeline.py does, then loads the hourly weather table it writes.

How to read the output: “Processing: 20250319_163237.WAV” means it found the raw recording. The table should have 3 rows (16:00, 17:00 and 18:00), because the team’s earlier fix fetches one extra hour at the end of a recording. A RuntimeError means weather couldn’t be fetched, and the new code stops on purpose. The usual cause is no internet. It needs a connection.

In [5]:
import inspect
import weather_matching
print("weather_match signature:", inspect.signature(weather_matching.weather_match))

matches_csv = OUTPUTS_DIR / "nb06_weather_matches.csv"
weather_matching.weather_match(
    weather_csv=str(weather_hourly_csv),
    metadata_csv=str(OUTPUTS_DIR / "nb05_metadata.csv"),
    output_csv=str(matches_csv),
)

m = pd.read_csv(matches_csv)
print("\nShape:", m.shape, "| unique clips:", m["clip_name"].nunique())
print("Blank weather values:", m[["Temperature", "Windspeed", "Humidity"]].isna().sum().to_dict())
m["clip_time"] = pd.to_datetime(m["clip_name"].str.extract(r"(\d{8}_\d{6})")[0], format="%Y%m%d_%H%M%S")
print(m.groupby("Temperature")["clip_time"].agg(["min", "max", "count"]).to_string())

weather_match signature: (weather_csv='path/to/weather_data.csv', metadata_csv='path/to/metadata.csv', output_csv='path/to/weather_matches.csv')



Shape: (1747, 12) | unique clips: 1747
Blank weather values: {'Temperature': 0, 'Windspeed': 0, 'Humidity': 0}
                            min                 max  count
Temperature                                               
26.1        2025-03-19 17:30:01 2025-03-19 17:59:55    599
27.1        2025-03-19 16:32:37 2025-03-19 17:29:58   1148


What this does: pairs each clip’s time (from Step 5’s metadata) with the hourly weather, using weather_match with its file paths as arguments, as run_pipeline.py does.

How to read the output: 1,747 rows and 0 blanks means every clip got weather. In the old code, 599 clips (34%) had none. If any clip has blank weather, the new code raises a ValueError and names example clips, instead of passing blanks on. The last table shows which hour’s weather each clip received, and shows the matching rule: nearest hour.

In [6]:
import os, tempfile, shutil

test_root = Path(tempfile.mkdtemp(prefix="lower_wav_test_"))
day = test_root / "Audio_Moth_5_031925"
day.mkdir()
os.symlink(RAW_DIR / "Audio_Moth_5_031925" / "20250319_163237.WAV", day / "20250319_163237.wav")
print("Files:", [p.name for p in day.iterdir()])

out = test_root / "weather_lower.csv"
try:
    weather_info.get_weather_info_all(audio_folder=str(test_root), output_csv=str(out))
    print("weather file written:", out.exists())
    if out.exists():
        print("Rows:", len(pd.read_csv(out)))
except Exception as e:
    print("FAILED:", type(e).__name__, e)
finally:
    shutil.rmtree(test_root)

Files: ['20250319_163237.wav']
Processing: 20250319_163237.wav
[OK] Combined weather data saved to /var/folders/18/gw6v9vc95bg8w0jxzmjwm2br0000gn/T/lower_wav_test_39j55br6/weather_lower.csv (3 hours)
weather file written: True
Rows: 3


**What this did:** linked the same raw recording under a lowercase name (`.wav`) in a temporary folder and ran the weather download on it. In the old code (4f86604), only uppercase `.WAV` raw files were found, and a lowercase file was ignored with only a warning.

**Result:** the lowercase file was found and processed ("Processing: 20250319_163237.wav"), and the weather file was written with 3 rows (16:00, 17:00 and 18:00). Step 6 now accepts both `.wav` and `.WAV`.

**Historical (code at 4f86604, before PR #2 and #3):** `weather_match()` failed with `KeyError: 'Datetime'`, 599 of 1,747 clips (34%) got blank weather, and a lowercase `.wav` raw file was ignored. PR #2 fixed the column names and the extra hour; PR #3 now stops with an error on blank weather. Results above are from the current code.

## Observations (Step 6, current code 9824a0d)
- `get_weather_info_all(audio_folder, output_csv)` and `weather_match(weather_csv, metadata_csv, output_csv)` take their paths as arguments, like `run_pipeline.py`. Needs internet.
- Download: 1 raw file in, 3 hourly rows out (16:00, 17:00, 18:00 on 2025-03-19; 27.8 to 26.1 °C, clear sky), 0.8 s. The extra 18:00 row comes from the fix for clips at the end of a recording.
- Match: 1,747 rows, 1,747 unique clips, 0 blank Temperature, Windspeed or Humidity. Matching is nearest-hour: clips 16:32:37 to 17:29:58 got the 17:00 weather (1,148), and clips from 17:30:01 on got the 18:00 weather (599). The 16:00 row is never used.
- Lowercase `.wav` raw files are now found (tested with a symlink: 3 rows written).
- The code stops with an error on blank weather (PR #3), instead of passing blanks on. Not tested: a failed download, such as without internet.
- Minor: the default argument values in both functions are still `path/to/...` placeholders. Harmless when called from `run_pipeline.py`.
- Historical (4f86604): `KeyError: 'Datetime'`, 599 clips with blank weather, and the lowercase-file miss are all fixed.